# AMML Ranking Competition 2026

Финальный пайплайн ранжирования русскоязычных фрагментов.
стекер обучается на большем числе запросов (AMML_STACK_QUERIES, по умолчанию 2400, порциями с кешем), энкодеры dity/modern при отсутствии ONNX считаются через PyTorch, 3 сида, шире сетка смеси.

**GPU:** gpu_scores.py считает zero-shot скоры и дообучает кросс-энкодер (OOF по темам) для всего train. Если в AMML_PRE_DIR лежат scores_<имя>.npz, ноутбук берёт их вместо ONNX на CPU; тогда ставьте AMML_STACK_QUERIES=0 (весь train) и AMML_EXTRA_ENCODERS=ft_bge,bge,dity (любые из посчитанных).


In [ ]:
!pip install -r requirements.txt

тут костыль, изначально делал через отдельный скрипт, но по условиям в репо не должно быть лишних файлов, поэтому вот так записал

In [ ]:
import os
import subprocess
from pathlib import Path

GPU_SCORES_SOURCE = r'''
"""GPU-скоринг для AMML: zero-shot скоры и дообучение кросс-энкодеров (OOF по темам).

Запуск (Colab/Kaggle с GPU, рядом train.csv и test.csv):
  python gpu_scores.py zeroshot jina mini dity modern bge
  python gpu_scores.py finetune bge
Результат: папка AMML_PRE_DIR (по умолчанию amml_pre) с файлами scores_<имя>.npz (ids, scores).
Ноутбук solution_stacked_v3 подхватывает их автоматически (см. AMML_PRE_DIR).
Окружение: AMML_DATA_DIR, AMML_MAXLEN (384/256), AMML_BS, AMML_LR, AMML_EPOCHS, AMML_FOLDS, AMML_QPS (запросов на шаг).
"""
import gc, html, os, random, re, sys, unicodedata
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.model_selection import GroupKFold
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

import transformers.models.xlm_roberta.modeling_xlm_roberta as _xlmr
if not hasattr(_xlmr, "create_position_ids_from_input_ids"):
    def _create_position_ids(input_ids, padding_idx, past_key_values_length=0):
        mask = input_ids.ne(padding_idx).int()
        return ((torch.cumsum(mask, dim=1).type_as(mask) + past_key_values_length) * mask).long() + padding_idx
    _xlmr.create_position_ids_from_input_ids = _create_position_ids

DATA = Path(os.environ.get("AMML_DATA_DIR", "."))
OUT = Path(os.environ.get("AMML_PRE_DIR", "amml_pre"))
MODELS = {
    "jina": "jinaai/jina-reranker-v2-base-multilingual",
    "mini": "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
    "dity": "DiTy/cross-encoder-russian-msmarco",
    "modern": "ARGA100/ru-reranker-modernbert-small",
    "bge": "BAAI/bge-reranker-v2-m3",
}
MAXLEN = int(os.environ.get("AMML_MAXLEN", "384"))
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("устройство:", DEV, flush=True)
ACCENT = re.compile("[̀́́​‌‍﻿]")
EMPTY = {"", "—", "-", "не указано", "n/a", "nan", "null", "none"}


@lru_cache(maxsize=None)
def clean(value, title=False):
    text = unicodedata.normalize("NFKC", html.unescape(value))
    text = unicodedata.normalize("NFC", ACCENT.sub("", text))
    text = re.sub(r"</?[a-zA-Z][^<>]{0,100}>|\[\s*\d{1,4}\s*\]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    if title:
        text = re.sub(r"\s*[-—|]\s*Википедия.*$", "", text, flags=re.I)
    return "" if text.casefold() in EMPTY else text


def clean_titles(frame):
    titles = frame["title"].map(lambda t: clean(t, True))
    observed = titles.mask(titles.eq(""))
    return observed.fillna(observed.groupby(frame["article_id"]).transform("first")).fillna("")


def load():
    train = pd.read_csv(DATA / "train.csv", dtype=str, keep_default_na=False)
    test = pd.read_csv(DATA / "test.csv", dtype=str, keep_default_na=False)
    train["relevance"] = train.relevance.astype(int)
    return train, test


def inputs(frame):
    titles, texts = clean_titles(frame), frame["text"].map(clean)
    passages = [f"{t}. {x}" if t else x for t, x in zip(titles, texts)]
    return frame["query"].map(clean).tolist(), passages


def topic_groups(frame):
    q, queries = pd.factorize(frame.query_id, sort=True)
    d, documents = pd.factorize(frame.document_id, sort=True)
    a, articles = pd.factorize(frame.article_id, sort=True)
    rare = frame.groupby("article_id").query_id.transform("nunique").le(2).to_numpy()
    nodes = len(queries) + len(documents) + len(articles)
    rows = np.concatenate([q, q[rare]])
    cols = np.concatenate([d + len(queries), a[rare] + len(queries) + len(documents)])
    graph = coo_matrix((np.ones(len(rows), dtype=np.int8), (rows, cols)), shape=(nodes, nodes))
    return connected_components(graph, directed=False)[1][q]


def to_score(logits):
    logits = logits.float().cpu().numpy()
    return logits[:, -1] - logits[:, 0] if logits.ndim == 2 and logits.shape[1] == 2 else logits.reshape(len(logits), -1)[:, 0]


def load_model(name, train_mode=False):
    tok = AutoTokenizer.from_pretrained(MODELS[name], trust_remote_code=True)
    model = AutoModelForSequenceClassification.from_pretrained(MODELS[name], trust_remote_code=True).to(DEV)
    if DEV == "cuda" and not train_mode:
        model = model.half()
    return tok, model


def predict(model, tok, queries, passages, bs, autocast=True):
    model.eval()
    order = np.argsort([len(p) for p in passages], kind="stable")
    out = np.zeros(len(passages), dtype=np.float32)
    for start in range(0, len(order), bs):
        rows = order[start:start + bs]
        batch = tok([queries[i] for i in rows], [passages[i] for i in rows], truncation="only_second",
                    max_length=MAXLEN, padding=True, return_tensors="pt")
        batch = {k: v.to(DEV) for k, v in batch.items() if k != "token_type_ids" or "token_type_ids" in model.forward.__code__.co_varnames}
        with torch.inference_mode(), torch.autocast(DEV, dtype=torch.bfloat16 if (DEV == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16, enabled=autocast and DEV == "cuda"):
            out[rows] = to_score(model(**batch).logits)
    return out


def save(name, ids, scores):
    OUT.mkdir(parents=True, exist_ok=True)
    assert np.isfinite(scores).all()
    np.savez(OUT / f"scores_{name}.npz", ids=np.asarray(ids, dtype=str), scores=scores.astype(np.float32))
    print("saved", name, len(scores), flush=True)


def zeroshot(name):
    train, test = load()
    tok, model = load_model(name)
    bs = int(os.environ.get("AMML_BS", "64"))
    parts = [predict(model, tok, *inputs(f), bs) for f in (train, test)]
    save(name, np.concatenate([train.id, test.id]), np.concatenate(parts))
    del model; gc.collect(); torch.cuda.empty_cache()


def listnet_loss(logits, labels):
    target = labels / labels.sum().clamp(min=1)
    return -(target * torch.log_softmax(logits, 0)).sum()


def finetune(name):
    train, test = load()
    folds, epochs = int(os.environ.get("AMML_FOLDS", "5")), int(os.environ.get("AMML_EPOCHS", "2"))
    lr, qps, bs = float(os.environ.get("AMML_LR", "1e-5")), int(os.environ.get("AMML_QPS", "4")), int(os.environ.get("AMML_BS", "32"))
    ml = int(os.environ.get("AMML_MAXLEN", "256"))
    global MAXLEN; MAXLEN = ml
    q_tr, p_tr = inputs(train); q_te, p_te = inputs(test)
    y = train.relevance.to_numpy(dtype=np.float32)
    oof = np.zeros(len(train), np.float32)
    use_bf16 = DEV == "cuda" and torch.cuda.is_bf16_supported()
    splits = list(GroupKFold(folds).split(train, groups=topic_groups(train)))
    done_folds, test_sum = set(), np.zeros(len(test), np.float32)
    partial_path = OUT / f"scores_ft_{name}_partial.npz"
    if partial_path.is_file():
        with np.load(partial_path, allow_pickle=False) as c:
            ids, sc = c["ids"], c["scores"]
        if len(ids) == len(train) + len(test):
            oof[:] = sc[:len(train)]
            done_folds = {i for i, (_, v) in enumerate(splits) if (oof[v] != 0).all()}
            test_sum = sc[len(train):] * max(len(done_folds), 1) if done_folds else test_sum
            print("продолжаю, готовые фолды:", sorted(done_folds), flush=True)
    for fold, (fit, val) in enumerate(splits):
        if fold in done_folds:
            continue
        random.seed(fold); torch.manual_seed(fold)
        tok, model = load_model(name, train_mode=True)
        try: model.gradient_checkpointing_enable()
        except Exception: pass
        qidx = list(train.iloc[fit].groupby("query_id", sort=False).indices.values())
        qidx = [fit[ix] for ix in qidx]
        steps = epochs * ((len(qidx) + qps - 1) // qps)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=.01)
        sched = get_linear_schedule_with_warmup(opt, int(.1 * steps), steps)
        scaler = torch.amp.GradScaler(enabled=DEV == "cuda" and not use_bf16)
        done = 0
        for epoch in range(epochs):
            random.shuffle(qidx); model.train()
            for s in range(0, len(qidx), qps):
                group = qidx[s:s + qps]
                rows = np.concatenate(group)
                batch = tok([q_tr[i] for i in rows], [p_tr[i] for i in rows], truncation="only_second",
                            max_length=ml, padding=True, return_tensors="pt")
                batch = {k: v.to(DEV) for k, v in batch.items() if k != "token_type_ids" or "token_type_ids" in model.forward.__code__.co_varnames}
                with torch.autocast(DEV, dtype=torch.bfloat16 if use_bf16 else torch.float16, enabled=DEV == "cuda"):
                    logits = model(**batch).logits.float().reshape(len(rows), -1)[:, -1]
                loss, offset = 0., 0
                for g in group:
                    loss = loss + listnet_loss(logits[offset:offset + len(g)], torch.from_numpy(y[g]).to(DEV)); offset += len(g)
                loss = loss / len(group)
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward(); scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.)
                scaler.step(opt); scaler.update(); sched.step(); done += 1
                if done % 100 == 0: print(f"fold {fold} step {done}/{steps} loss {loss.item():.4f}", flush=True)
        oof[val] = predict(model, tok, [q_tr[i] for i in val], [p_tr[i] for i in val], bs)
        test_sum += predict(model, tok, q_te, p_te, bs)
        done_folds.add(fold)
        print(f"fold {fold} done", flush=True)
        del model, opt; gc.collect(); torch.cuda.empty_cache()
        save(f"ft_{name}_partial", np.concatenate([train.id, test.id]), np.concatenate([oof, test_sum / len(done_folds)]))
    save(f"ft_{name}", np.concatenate([train.id, test.id]), np.concatenate([oof, test_sum / len(done_folds)]))
    (OUT / f"scores_ft_{name}_partial.npz").unlink(missing_ok=True)


if __name__ == "__main__":
    mode, names = sys.argv[1], sys.argv[2:]
    for n in names:
        {"zeroshot": zeroshot, "finetune": finetune}[mode](n)

'''

script = Path("gpu_scores.py")
script.write_text(GPU_SCORES_SOURCE, encoding="utf-8")
print("gpu_scores.py записан:", script.resolve())

import torch
has_gpu = torch.cuda.is_available()
print("GPU доступен:", has_gpu)

RUN_GPU_SCORES = has_gpu and os.environ.get("AMML_RUN_GPU_SCORES", "1") == "1"

if RUN_GPU_SCORES:
    env = os.environ.copy()
    env.setdefault("AMML_DATA_DIR", str(Path(".").resolve()))
    env.setdefault("AMML_PRE_DIR", "amml_pre")
    env.setdefault("AMML_MAXLEN", "256")

    cmd = ["python", "gpu_scores.py", "zeroshot", "jina", "mini", "dity", "modern", "bge"]
    print("запуск:", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True, env=env)

    if os.environ.get("AMML_FINETUNE", "0") == "1":
        cmd = ["python", "gpu_scores.py", "finetune", "bge"]
        print("запуск:", " ".join(cmd), flush=True)
        subprocess.run(cmd, check=True, env=env)

    print("готово, файлы в", Path(env["AMML_PRE_DIR"]).resolve())
else:
    print("GPU-скоринг пропущен — ноутбук посчитает скоры на CPU (ONNX/PyTorch).")

In [5]:
import os
import platform
from pathlib import Path

import numpy as np
import pandas as pd

BASELINE_3 = 0.932861
OUTPUT_DIR = Path(os.environ.get("AMML_OUTPUT_DIR","/kaggle/working" if Path("/kaggle/working").is_dir() else "."))
MODEL_DIR = Path(os.environ.get("AMML_MODEL_DIR",str(OUTPUT_DIR/"amml_models")))
CACHE_DIR = Path(os.environ.get("AMML_CACHE_DIR",str(OUTPUT_DIR/"amml_cache")))


def load_data():
    names = [("train.csv","train (3).csv"),("test.csv",),("sample_submission.csv","sample_submission (2).csv")]
    configured = os.environ.get("AMML_DATA_DIR")
    roots = [Path(configured)] if configured else [Path("data"),Path("."),Path("/kaggle/input/amml-ranking-competition-2026"),Path("/kaggle/input/competitions/amml-ranking-competition-2026")]
    if not configured and Path("/kaggle/input").is_dir():
        roots.extend(path.parent for path in sorted(Path("/kaggle/input").rglob("sample_submission.csv")))
    for root in dict.fromkeys(roots):
        paths = [next((root/n for n in options if (root/n).is_file()),None) for options in names]
        if all(paths):
            break
    else:
        raise FileNotFoundError("Укажите папку с train, test и sample_submission в AMML_DATA_DIR.")
    train,test,sample=(pd.read_csv(path,dtype=str,keep_default_na=False) for path in paths)
    train["relevance"]=pd.to_numeric(train.relevance,errors="raise").astype("int8")
    if set(train.relevance.unique())!={0,1} or not all(frame.id.is_unique for frame in (train,test,sample)):
        raise ValueError("Ожидаются бинарные метки train и уникальные id.")
    if (set(train.query_id)&set(test.query_id) or list(sample.columns)!=["id","prediction"]
        or set(test.id)!=set(sample.id)):
        raise ValueError("Проверьте разделение запросов и совпадение id теста с шаблоном.")
    return train,test,sample


def write_submission(test,sample,prediction):
    prediction=np.asarray(prediction,dtype=np.float64)
    if prediction.shape!=(len(test),) or not np.isfinite(prediction).all():
        raise ValueError("Нужен один конечный скор на каждую тестовую пару.")
    scores=pd.Series(prediction,index=test.id)
    out=sample[["id"]].assign(prediction=sample.id.map(scores))
    OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
    destination=OUTPUT_DIR/"submission.csv"
    out.to_csv(destination,index=False)
    print(f"{len(out):,} строк → {destination.resolve()}")
    return out

## Предобработка и признаки

In [6]:
import html
import math
import re
import unicodedata
from collections import Counter
from functools import lru_cache

import numpy as np
import pandas as pd
import pymorphy3
from rapidfuzz.fuzz import ratio, token_set_ratio
from sklearn.feature_extraction.text import TfidfVectorizer

MORPH = pymorphy3.MorphAnalyzer()
WORD = re.compile(r"[^\W_]+")
ACCENT = re.compile("[\u0300\u0301\u0341\u200b\u200c\u200d\ufeff]")
EMPTY = {"", "—", "-", "не указано", "n/a", "nan", "null", "none"}
STOP = set("""а без более был была были было быть в вам вас весь во вот все всего вы где да для до его ее если есть еще же за из или им их к как какая какие каким какими каких какое какой какую какова каков каковы кем ко когда кого которой кому которая которые который которых кто куда ли между мне может можно мы на над надо назовите найти находиться наш не него нее нет ни них но ну о об один она они оно от отчего перед по под почему при про с сам свое себя сколько со так такой там те тем то того тоже той только том тот тут ты у уже хотя чем чему через что чтобы чья чье это этот я являлся является являться называться назывался звали зовут назвать известный известен знать""".split())
PATTERNS = [
    ("когда год дата лет век", r"\b(?:[12][0-9]{3}|\d{1,2}\s+(?:января|февраля|марта|апреля|мая|июня|июля|августа|сентября|октября|ноября|декабря))\b"),
    ("площадь размер длина высота вес глубина ширина расстояние объем", r"\d[\d,. ]*\s*(?:км|мм|см|метр|кг|тонн|гектар|километр|м²|квадрат|литр)"),
    ("родился родилась родиться", r"родил[а-я]*|род\.|появил[а-я]* на свет"),
    ("умер умереть умерла погиб погибнуть смерть", r"умер[а-я]*|погиб[а-я]*|скончал[а-я]*|смерт[а-я]*|убит[а-я]*"),
    ("основать основал основали основан создать создал создан открыли открыть открытие", r"основа[а-я]*|учреди[а-я]*|созда[а-я]*|открыл[а-я]*|открыти[а-я]*|появил[а-я]*"),
    ("население жителей численность", r"населени[а-я]*|жител[а-я]*|прожива[а-я]*|человек"),
    ("где находится расположен расположена местоположение", r"располож[а-я]*|наход[а-я]*|территори[а-я]*|местополож[а-я]*|\bгород[а-я]*"),
    ("перевод переводится значение означает обозначает", r"перевод[а-я]*|означа[а-я]*|обознача[а-я]*|букваль[а-я]*"),
    ("сколько", r"\b\d[\d.,]*\b"),
]
PATTERNS = [(set(terms.split()), re.compile(pattern, re.I)) for terms, pattern in PATTERNS]


@lru_cache(maxsize=120_000)
def clean(value, title=False):
    text = unicodedata.normalize("NFKC", html.unescape(value))
    text = unicodedata.normalize("NFC", ACCENT.sub("", text))
    text = re.sub(r"</?[a-zA-Z][^<>]{0,100}>|\[\s*\d{1,4}\s*\]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    if title:
        text = re.sub(r"\s*[-—|]\s*Википедия.*$", "", text, flags=re.I)
    return "" if text.casefold() in EMPTY else text


@lru_cache(maxsize=190_000)
def lemma(token):
    if re.fullmatch(r"[а-я]{2,}", token):
        return MORPH.parse(token)[0].normal_form.replace("ё", "е")
    return token


def clean_titles(frame):
    titles = frame["title"].map(lambda t: clean(t, True))
    observed = titles.mask(titles.eq(""))
    return observed.fillna(observed.groupby(frame["article_id"]).transform("first")).fillna("")


@lru_cache(maxsize=85_000)
def tokens(text):
    raw = WORD.findall(text)
    word = [w.lower().replace("ё", "е") for w in raw]
    lemmas = [lemma(w) for w in word]
    return raw, word, lemmas


def fit_lexical(train):
    passages = train.loc[~train.document_id.duplicated(), "text"].map(clean)
    freq = Counter()
    for passage in passages:
        freq.update(set(tokens(passage)[2]))
    idf = {word: math.log1p((len(passages)-n+0.5)/(n+0.5)) for word,n in freq.items()}
    corpus = pd.concat([passages, train["query"].map(clean).drop_duplicates(), clean_titles(train).drop_duplicates()])
    char_vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), min_df=2,
                                     max_features=170_000, sublinear_tf=True, dtype=np.float32)
    return idf, char_vectorizer.fit(corpus)


def overlap(query, target, idf, name):
    terms = (set(query)-STOP) or set(query)
    counts = Counter(w for w in target if w in terms)
    positions = [(i,w) for i,w in enumerate(target) if w in counts]
    weights = {w:idf.get(w,9.5) for w in terms}
    total = sum(weights.values()) or 1.
    hit = sum(weights[w] for w in counts)
    out = {
        "coverage": len(counts)/max(len(terms),1), "weighted": hit/total,
        "missing": max((weights[w] for w in terms-counts.keys()), default=0),
        "best": max((weights[w] for w in counts), default=0),
        "bm25": sum(weights[w]*2.2*n/(n+1.2*(.3+.7*len(target)/95)) for w,n in counts.items()),
        "first": positions[0][0]/max(1,len(target)) if positions else 1,
        "head": sum(weights[w] for w in terms.intersection(target[:30]))/total,
        "freq": sum(counts.values())/max(len(target),1),
    }
    for n in (2,3):
        q = set(zip(*(query[i:] for i in range(n))))
        out[f"ngram{n}"] = len(q.intersection(zip(*(target[i:] for i in range(n)))))/max(1,len(q))
    best, close = 0., 0.
    for step,(start,_) in enumerate(positions):
        local = set()
        weight = 0.
        for end,term in positions[step:]:
            if end-start>22:break
            if term not in local:
                local.add(term)
                weight += weights[term]
            best = max(best,weight)
            close = max(close,weight/(1+(end-start)/11))
    out["window"],out["close"]=best/total,close/total
    return {f"{name}_{key}":v for key,v in out.items()}


def rank_features(frame, lexical):
    idf, vectorizer = lexical
    queries = frame["query"].map(clean)
    titles = clean_titles(frame)
    texts = frame["text"].map(clean)
    rows = []
    for query,title,text in zip(queries,titles,texts):
        qraw,qwords,ql = tokens(query)
        traw,tw,tl = tokens(title)
        praw,pw,pl = tokens(text)
        qi,ti,pi = set(ql)-STOP,set(tl)-STOP,set(pl)-STOP
        entities = {lemma(x.lower().replace("ё","е")) for x in qraw if x[0].isupper()}-STOP
        pcaps = {lemma(x.lower().replace("ё","е")) for x in praw if x[0].isupper()}
        q_phrase,t_phrase = " ".join(qwords)," ".join(tw)
        sentences = re.split(r"(?<=[.!?])\s+(?=[«(\"А-ЯA-Z0-9Ё])",text)
        sentence_scores = [sum(idf.get(t,9.5) for t in qi.intersection(tokens(s)[2])) for s in sentences]
        row = {
            "q_first": qwords[0] if qwords else "",
            "q_lemma": ql[0] if ql else "",
            "q_words": len(ql), "t_words": len(tl), "p_words": len(pl),
            "p_chars": len(text), "p_unique": len(set(pl))/max(1,len(pl)),
            "p_sentences": len(sentences), "p_numbers": sum(x.isdigit() for x in pw),
            "p_years": sum(x.isdigit() and len(x)==4 for x in pw), "p_brackets": text.count("("),
            "p_commas": text.count(","), "p_clauses":text.count(";"), "p_capitals":len(pcaps),
            "entity_p":len(entities&pi)/max(1,len(entities)), "entity_t":len(entities&ti)/max(1,len(entities)),
            "entity_p_caps":len(entities&pcaps)/max(1,len(entities)),
            "union":len(qi&(ti|pi))/max(1,len(qi)), "triple":len(qi&ti&pi)/max(1,len(qi)),
            "title_covered_q":len(ti&qi)/max(1,len(ti)), "title_covered_p":len(ti&pi)/max(1,len(ti)),
            "p_title_excluded_q":len((qi&pi)-ti)/max(1,len(qi)),
            "s_best":max(sentence_scores,default=0), "s_first":sentence_scores[0],
            "s_pos":int(np.argmax(sentence_scores))/max(1,len(sentence_scores)),
            "title_in_query": int(bool(t_phrase and t_phrase in q_phrase)),
            "title_fuzzy_match":token_set_ratio(q_phrase,t_phrase)/100,
            "title_similar_token": max((ratio(q,t)/100 for q in qi for t in ti if min(len(q),len(t))>=4),default=0),
            "years_match":len({q for q in qwords if q.isdigit()}&set(pw)),
        }
        for q,t,n in [(ql,pl,"plem"),(ql,tl,"tlem"),(qwords,pw,"pword"),(qwords,tw,"tword")]:
            row.update(overlap(q,t,idf,n))
        for i,(markers,pattern) in enumerate(PATTERNS):
            matches=list(pattern.finditer(text))
            row[f"type{i}_q"]=int(bool(markers.intersection(qwords+ql)))
            row[f"type{i}_p"]=len(matches)
            row[f"type{i}_first"]=matches[0].start()/max(1,len(text)) if matches else np.nan
        rows.append(row)
    out = pd.DataFrame(rows,index=frame.index)
    qid, aid, did = frame.query_id,frame.article_id,frame.document_id
    out["source"] = frame.ingest_source
    for field in ("query","title","text"):
        raw=frame[field]
        for name, pat in [("acute","[\u0300\u0301]"),("nbsp","[\xa0\u202f]"),("line","\n"),("upper",r"\b[А-ЯЁA-Z]{2,5}\b"),("wiki",r"икипедия"),("html",r"&lt;|<[a-zA-Z]+>")]:
            out[f"{field}_{name}"]=raw.str.count(pat)
    out["title_missing"]=frame.title.map(lambda t: not clean(t,True))
    for field in ("passage_index","article_passage_count","article_char_count"):
        v = pd.to_numeric(frame[field].str.replace(r"\s+","",regex=True).str.replace(",","."),errors="coerce")
        median = v.where(v>=0).groupby(aid).transform("median")
        out[field] = v.where(v>=0) if field=="passage_index" else v.where(v>=0).fillna(median)
        out[field+"_missing"] = v.isna()
        out[field+"_invalid"] = v.lt(0)
        out[field+"_dev"] = (v-median).abs()/median.clip(lower=1)
        out[field+"_format"] = frame[field].str.contains(r"[,\s]")
    out["rel_index"] = out.passage_index/out.article_passage_count.clip(lower=1)
    out["avg_length"] = out.article_char_count/out.article_passage_count.clip(lower=1)
    out["rel_length"] = out.p_chars/out.avg_length.clip(lower=1)
    out["q_count"] = qid.groupby(qid).transform("size")
    out["q_articles"] = aid.groupby(qid).transform("nunique")
    out["q_a_count"] = qid.groupby([qid,aid]).transform("size")
    out["a_q_count"] = qid.groupby(aid).transform("nunique")
    out["d_q_count"] = qid.groupby(did).transform("nunique")
    out["a_d_count"] = did.groupby(aid).transform("nunique")
    out["q_source_count"] = qid.groupby([qid,frame.ingest_source]).transform("size")
    out["a_q_other"] = aid.groupby(aid).transform("size") - out.q_a_count

    cos = {key:[] for key in ("char_qp","char_qt","char_tp","char_head")}
    for start in range(0,len(frame),1600):
        ix=slice(start,start+1600)
        q=vectorizer.transform(queries.iloc[ix])
        t=vectorizer.transform(titles.iloc[ix])
        p=vectorizer.transform(texts.iloc[ix])
        h=vectorizer.transform(texts.iloc[ix].str.slice(0,280))
        for name,left,right in [("char_qp",q,p),("char_qt",q,t),("char_tp",t,p),("char_head",q,h)]:
            cos[name].append(np.asarray(left.multiply(right).sum(axis=1)).ravel())
    for name,blocks in cos.items():
        out[name]=np.concatenate(blocks)

    relative={}
    for name in ["plem_weighted","plem_bm25","plem_window","plem_close","tlem_weighted","tlem_bm25","pword_bm25","tword_bm25","s_best","char_qp","char_qt","char_head","title_covered_q","entity_p","entity_p_caps","title_fuzzy_match","union","p_chars","passage_index","rel_index","article_char_count","article_passage_count","q_a_count","d_q_count","text_acute","a_q_other"]:
        value=out[name]
        group=value.groupby(qid)
        article=value.groupby([qid,aid])
        relative[name+"_q_rank"]=group.rank(pct=True)
        relative[name+"_q_gap"]=group.transform("max")-value
        relative[name+"_a_mean"]=article.transform("mean")
        relative[name+"_a_max"]=article.transform("max")
        relative[name+"_a_gap"]=relative[name+"_a_max"]-value
    out=pd.concat([out,pd.DataFrame(relative,index=frame.index)],axis=1)
    cats=["source","q_first","q_lemma"]
    cols=out.columns.difference(cats)
    out[cols]=out[cols].astype(np.float32).replace([np.inf,-np.inf],np.nan)
    return out

## Семантические скоры

In [ ]:
import gc
import hashlib
import os
import platform
import tempfile
import urllib.request
from pathlib import Path

import numpy as np
import onnxruntime as ort
from tokenizers import Tokenizer

ENCODERS = {
    "jina": (
        "jinaai/jina-reranker-v2-base-multilingual",
        "9cfeff2df7d40d1b78e75e5e9cebec92a99813c9",
        "model_quantized.onnx",
        14,
    ),
    "mini": (
        "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
        "1427fd652930e4ba29e8149678df786c240d8825",
        "model_qint8_arm64.onnx" if platform.machine().lower() in {"arm64","aarch64"} else "model_quint8_avx2.onnx",
        16,
    ),
}
ONNX_CANDIDATES = ("onnx/model_quantized.onnx", "onnx/model.onnx", "model.onnx")
ENCODERS["dity"] = ("DiTy/cross-encoder-russian-msmarco", "main", ONNX_CANDIDATES, 16)
ENCODERS["modern"] = ("ARGA100/ru-reranker-modernbert-small", "main", ONNX_CANDIDATES, 24)
MAX_TOKENS = int(os.environ.get("AMML_MAX_TOKENS", "192"))


def download_encoder_file(name, filename):
    repo, revision, _, _ = ENCODERS[name]
    folder = MODEL_DIR / name
    folder.mkdir(parents=True, exist_ok=True)
    destination = folder / filename.rsplit("/", 1)[-1]
    if destination.is_file():
        return destination
    with tempfile.NamedTemporaryFile(dir=folder, delete=False) as stream:
        temporary = Path(stream.name)
        url = f"https://huggingface.co/{repo}/resolve/{revision}/{filename}?download=true"
        try:
            with urllib.request.urlopen(url, timeout=180) as response:
                while block := response.read(4 * 1024 * 1024):
                    stream.write(block)
            stream.close()
            temporary.replace(destination)
        finally:
            temporary.unlink(missing_ok=True)
    return destination


def fetch_model(name):
    """Ищет ONNX-файл модели: у новых энкодеров расположение файла может отличаться."""
    option = ENCODERS[name][2]
    if isinstance(option, str):
        return download_encoder_file(name, f"onnx/{option}")
    errors = []
    for path in option:
        try:
            return download_encoder_file(name, path)
        except Exception as error:
            errors.append(f"{path}: {error}")
    raise RuntimeError(f"ONNX для {name} не найден ({'; '.join(errors)})")


def encoder_cache_key(frame, name, batch_size=None):
    batch_size = batch_size or ENCODERS[name][3]
    digest = hashlib.sha256(repr((ENCODERS[name][:3], MAX_TOKENS, batch_size, "clean-title-v1")).encode())
    content = pd.util.hash_pandas_object(frame[["id", "query", "title", "text", "article_id"]], index=False)
    digest.update(content.to_numpy().tobytes())
    return f"{name}_{digest.hexdigest()[:24]}.npz"


def save_scores(destination, ids, scores):
    with tempfile.NamedTemporaryFile(dir=destination.parent, suffix=".npz", delete=False) as pending:
        temporary = Path(pending.name)
    try:
        np.savez(temporary, ids=ids, scores=scores)
        temporary.replace(destination)
    finally:
        temporary.unlink(missing_ok=True)


def read_scores(path, ids):
    with np.load(path, allow_pickle=False) as cached:
        scores = cached["scores"]
        if not np.array_equal(cached["ids"], ids) or scores.shape != ids.shape or not np.isfinite(scores).all():
            raise ValueError(f"Неверный кеш для {path.name}.")
    return scores


def encoder_scores(name, frame, batch_size=None):
    batch_size = batch_size or ENCODERS[name][3]
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    cache = CACHE_DIR / encoder_cache_key(frame, name, batch_size)
    ids = frame.id.to_numpy(dtype=str)
    if cache.is_file():
        scores = read_scores(cache, ids)
        print(f"{name}: {len(scores):,} сохранённых скоров", flush=True)
        return scores

    vocabulary = Tokenizer.from_file(str(download_encoder_file(name, "tokenizer.json")))
    vocabulary.enable_truncation(max_length=MAX_TOKENS, strategy="only_second")
    titles, texts = clean_titles(frame), frame["text"].map(clean)
    passages = [f"{title}. {text}" if title else text for title, text in zip(titles, texts)]
    encoded = vocabulary.encode_batch(list(zip(frame["query"].map(clean), passages)))
    lengths = np.asarray([len(e.ids) for e in encoded])
    order = np.argsort(lengths, kind="stable")
    pad = vocabulary.token_to_id("<pad>")
    if pad is None:
        pad = vocabulary.token_to_id("[PAD]") or 0

    options = ort.SessionOptions()
    options.intra_op_num_threads = int(os.environ.get("AMML_THREADS", "4"))
    options.inter_op_num_threads = 1
    options.enable_cpu_mem_arena = False
    options.enable_mem_pattern = False
    options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    options.add_session_config_entry("session.intra_op.allow_spinning", "0")
    model = ort.InferenceSession(str(fetch_model(name)), sess_options=options, providers=["CPUExecutionProvider"])
    input_names = {item.name for item in model.get_inputs()}
    partial = cache.with_suffix(".partial.npz")
    scores = np.empty(len(frame), dtype=np.float32)
    completed = 0
    if partial.is_file():
        with np.load(partial, allow_pickle=False) as cached:
            completed = len(cached["ids"])
        saved_rows = order[:completed]
        scores[saved_rows] = read_scores(partial, ids[saved_rows])
    for start in range(completed, len(frame), batch_size):
        rows = order[start:start + batch_size]
        width = min(MAX_TOKENS, 8 * ((int(lengths[rows].max()) + 7) // 8))
        input_ids = np.full((len(rows), width), pad, dtype=np.int64)
        mask = np.zeros_like(input_ids)
        segments = np.zeros_like(input_ids)
        for row, frame_index in enumerate(rows):
            size = lengths[frame_index]
            input_ids[row, :size] = encoded[frame_index].ids
            mask[row, :size] = 1
            segments[row, :size] = encoded[frame_index].type_ids
        feed = {"input_ids": input_ids, "attention_mask": mask, "token_type_ids": segments}
        logits = model.run(None, {key: value for key, value in feed.items() if key in input_names})[0]
        scores[rows] = logits[:, -1] - logits[:, 0] if logits.ndim == 2 and logits.shape[1] == 2 else logits.ravel()
        end = start + len(rows)
        if (start-completed) % (64 * batch_size) == 0 or end == len(frame):
            done = order[:end]
            save_scores(partial, ids[done], scores[done])
            print(f"{name}: {end:,}/{len(frame):,}", flush=True)
    if not np.isfinite(scores).all():
        raise ValueError(f"Некорректный результат кодировщика {name}.")
    save_scores(cache, ids, scores)
    partial.unlink(missing_ok=True)
    del model
    gc.collect()
    return scores


_onnx_scores = encoder_scores


def torch_scores(name, frame, batch_size=None):
    import inspect
    import torch
    from transformers import AutoModelForSequenceClassification
    batch_size = batch_size or ENCODERS[name][3]
    cache = CACHE_DIR / encoder_cache_key(frame, name, batch_size)
    ids = frame.id.to_numpy(dtype=str)
    if cache.is_file():
        return read_scores(cache, ids)
    vocabulary = Tokenizer.from_file(str(download_encoder_file(name, "tokenizer.json")))
    vocabulary.enable_truncation(max_length=MAX_TOKENS, strategy="only_second")
    titles, texts = clean_titles(frame), frame["text"].map(clean)
    passages = [f"{t}. {x}" if t else x for t, x in zip(titles, texts)]
    encoded = vocabulary.encode_batch(list(zip(frame["query"].map(clean), passages)))
    lengths = np.asarray([len(e.ids) for e in encoded])
    order = np.argsort(lengths, kind="stable")
    pad = vocabulary.token_to_id("<pad>")
    if pad is None:
        pad = vocabulary.token_to_id("[PAD]") or 0
    torch.set_num_threads(int(os.environ.get("AMML_THREADS", "4")))
    model = AutoModelForSequenceClassification.from_pretrained(ENCODERS[name][0]).eval()
    if os.environ.get("AMML_TORCH_QUANT", "1") == "1":
        model = torch.quantization.quantize_dynamic(model, {torch.nn.Linear}, dtype=torch.qint8)
    accepted = set(inspect.signature(model.forward).parameters)
    partial = cache.with_suffix(".partial.npz")
    scores = np.empty(len(frame), dtype=np.float32)
    completed = 0
    if partial.is_file():
        with np.load(partial, allow_pickle=False) as cached:
            completed = len(cached["ids"])
        scores[order[:completed]] = read_scores(partial, ids[order[:completed]])
    for start in range(completed, len(frame), batch_size):
        rows = order[start:start + batch_size]
        width = int(lengths[rows].max())
        input_ids = np.full((len(rows), width), pad, dtype=np.int64)
        mask = np.zeros_like(input_ids)
        segments = np.zeros_like(input_ids)
        for row, index in enumerate(rows):
            size = lengths[index]
            input_ids[row, :size] = encoded[index].ids
            mask[row, :size] = 1
            segments[row, :size] = encoded[index].type_ids
        feed = {"input_ids": input_ids, "attention_mask": mask, "token_type_ids": segments}
        with torch.inference_mode():
            logits = model(**{k: torch.from_numpy(v) for k, v in feed.items() if k in accepted}).logits.float().numpy()
        scores[rows] = logits[:, -1] - logits[:, 0] if logits.ndim == 2 and logits.shape[1] == 2 else logits.reshape(len(rows), -1)[:, 0]
        end = start + len(rows)
        if (start - completed) % (64 * batch_size) == 0 or end == len(frame):
            done = order[:end]
            save_scores(partial, ids[done], scores[done])
            print(f"{name} (torch): {end:,}/{len(frame):,}", flush=True)
    if not np.isfinite(scores).all():
        raise ValueError(f"Некорректный результат кодировщика {name}.")
    save_scores(cache, ids, scores)
    partial.unlink(missing_ok=True)
    del model
    gc.collect()
    return scores


PRE_DIR = Path(os.environ.get("AMML_PRE_DIR", "amml_pre"))


def precomputed_scores(name, frame):
    path = PRE_DIR / f"scores_{name}.npz"
    if not path.is_file():
        return None
    with np.load(path, allow_pickle=False) as cached:
        lookup = pd.Series(cached["scores"], index=cached["ids"])
    found = lookup.reindex(frame.id.to_numpy(dtype=str))
    if found.isna().any():
        raise ValueError(f"В {path.name} нет скоров для {int(found.isna().sum())} строк.")
    print(f"{name}: скоры из {path}", flush=True)
    return found.to_numpy(np.float32)


def encoder_scores(name, frame, batch_size=None):
    ready = precomputed_scores(name, frame)
    if ready is not None:
        return ready
    try:
        return _onnx_scores(name, frame, batch_size)
    except RuntimeError as error:
        if "ONNX для" not in str(error):
            raise
        print(f"{name}: готового ONNX нет, считаю через PyTorch", flush=True)
        return torch_scores(name, frame, batch_size)


## Обучение ранжировщика и выгрузка предсказаний

In [10]:
def within_query(frame, pred):
    pred=pd.Series(np.asarray(pred),index=frame.index)
    group=pred.groupby(frame.query_id)
    return ((pred-group.transform("mean"))/group.transform("std").fillna(1).clip(lower=1e-6)).to_numpy()


import numpy as np
import pandas as pd
from catboost import CatBoostClassifier,Pool


def train_lexical_model(train,target,fit_features,target_features):
    model=CatBoostClassifier(
        iterations=700,depth=6,learning_rate=.052,thread_count=3,
        allow_writing_files=False,verbose=False,random_seed=2026,
    )
    model.fit(Pool(fit_features,train.relevance,cat_features=["source","q_first","q_lemma"]))
    return model.predict(target_features,prediction_type="RawFormulaVal")

In [11]:
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.model_selection import GroupShuffleSplit

def strict_split(frame,seed=2026):
    q,queries=pd.factorize(frame.query_id,sort=True)
    d,documents=pd.factorize(frame.document_id,sort=True)
    nodes=len(queries)+len(documents)
    graph=coo_matrix((np.ones(len(q),dtype=np.int8),(q,d+len(queries))),shape=(nodes,nodes))
    _,groups=connected_components(graph,directed=False)
    stats=frame.groupby("query_id").relevance.agg(["size","sum"])
    stats["group"]=pd.Series(groups[q],index=frame.index).groupby(frame.query_id).first()
    val_indices=min((ix for _,ix in GroupShuffleSplit(n_splits=160,test_size=.25,random_state=seed).split(stats,groups=stats.group)),key=lambda ix:6*abs(len(ix)/len(stats)-.25)+.15*abs(stats.iloc[ix]["size"].mean()-stats["size"].mean())+.2*abs(stats.iloc[ix]["sum"].mean()-stats["sum"].mean()))
    valid=frame.query_id.isin(stats.index[val_indices])
    return frame[~valid],frame[valid]

def calibration_rows(train):
    fit, _ = strict_split(train, seed=2026)
    query_ids = fit.query_id.drop_duplicates()
    sampled = query_ids.sample(min(400, len(query_ids)), random_state=2026)
    return fit[fit.query_id.isin(sampled)]

import numpy as np
from scipy.optimize import minimize
from scipy.special import expit

CALIBRATION_COLUMNS = (
    "text_acute", "q_a_count", "char_qt", "plem_weighted",
    "entity_t", "p_chars", "rel_index", "d_q_count",
)


def calibration_matrix(frame, features):
    return np.column_stack([
        within_query(frame, features[name].fillna(0))
        for name in CALIBRATION_COLUMNS
    ])


def fit_jina_calibration(train, features, jina_scores):
    base = within_query(train, jina_scores)
    x = calibration_matrix(train, features)
    y = train.relevance.to_numpy()
    feature_gaps, score_gaps, weights = [], [], []
    for rows in train.groupby("query_id", sort=False).indices.values():
        positives = rows[y[rows] == 1]
        negatives = rows[y[rows] == 0]
        if not len(positives) or not len(negatives):
            continue
        pairs_pos = np.repeat(positives, len(negatives))
        pairs_neg = np.tile(negatives, len(positives))
        feature_gaps.append(x[pairs_pos] - x[pairs_neg])
        score_gaps.append(base[pairs_pos] - base[pairs_neg])
        weights.append(np.full(len(pairs_pos), 1 / len(pairs_pos)))
    pair_features = np.vstack(feature_gaps)
    pair_scores = np.concatenate(score_gaps)
    pair_weights = np.concatenate(weights)
    pair_weights /= pair_weights.sum()
    scale, l2 = .32, .28

    def objective(coefficients):
        gaps = (pair_scores + pair_features @ coefficients) / scale
        loss = np.dot(pair_weights, np.logaddexp(0, -gaps)) + .5 * l2 * np.dot(coefficients, coefficients)
        gradient = pair_features.T @ (-pair_weights * expit(-gaps)) / scale + l2 * coefficients
        return loss, gradient

    fitted = minimize(
        objective, np.zeros(len(CALIBRATION_COLUMNS)), method="L-BFGS-B", jac=True,
        bounds=[(-.25, .25)] * len(CALIBRATION_COLUMNS),
    )
    if not fitted.success:
        raise RuntimeError(f"Не удалось обучить поправку ранжирования: {fitted.message}")
    return fitted.x


def calibrated_ensemble(frame, features, jina, mini, lexical, correction):
    z_jina = within_query(frame, jina)
    blend = .85 * z_jina + .07 * within_query(frame, mini) + .08 * within_query(frame, lexical)
    calibrated_jina = z_jina + calibration_matrix(frame, features) @ correction
    return .5 * blend + .5 * calibrated_jina

## Стекинг поверх нейросетевых скоров
На выборке train (AMML_STACK_QUERIES, по умолчанию 1200 запросов) обучается CatBoost, который видит:
- скоры энкодеров (jina, mini и, если ONNX-файлы удалось скачать, dity и modern) и их контекст: ранг в запросе, отставание от лучшего, максимум/среднее/ранг по статье, softmax, число сильных кандидатов;
- похожесть фрагмента на кандидатов, которых нейросети считают релевантными (PRF);
- честный out-of-fold лексический скор и мета-признаки.

CatBoostRanker (YetiRank) останавливается по NDCG:top=10 на отложенных темах; рядом обучается CatBoostClassifier. Валидация — GroupKFold` по темам; веса смеси (классификатор внутри стекера, стекер против исходного ансамбля) выбираются по OOF. Энкодеры из AMML_EXTRA_ENCODERS (по умолчанию dity,modern; пустая строка - отключить) при ошибке загрузки пропускаются.

In [ ]:
import time
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from catboost import CatBoostRanker

STACK_QUERIES = int(os.environ.get("AMML_STACK_QUERIES", "2400"))
STACK_SEEDS = (11, 22, 33)
STACK_FEATURES = list(CALIBRATION_COLUMNS) + [
    "passage_index", "article_passage_count", "article_char_count", "title_fuzzy_match", "char_qp",
    "plem_bm25", "tlem_weighted", "entity_p", "s_best", "a_q_other", "title_covered_q", "plem_window",
]


def mean_ndcg(frame, prediction, k=10):
    """Средний по запросам NDCG@k с бинарной релевантностью."""
    pred = np.asarray(prediction, dtype=np.float64)
    total = 0.
    discount = 1 / np.log2(np.arange(2, k + 2))
    for rows in frame.groupby("query_id", sort=False).indices.values():
        y = frame.relevance.to_numpy()[rows]
        top = y[np.argsort(-pred[rows], kind="stable")][:k]
        ideal = np.sort(y)[::-1][:k]
        total += (top * discount[:len(top)]).sum() / max((ideal * discount[:len(ideal)]).sum(), 1e-9)
    return total / frame.query_id.nunique()


def topic_groups(frame):
    """Группы для валидации: запросы, связанные общими документами или редкими статьями, всегда в одной группе
    (в test статьи почти не пересекаются с train, обычное разбиение по запросам завышает метрику)."""
    q, queries = pd.factorize(frame.query_id, sort=True)
    d, documents = pd.factorize(frame.document_id, sort=True)
    a, articles = pd.factorize(frame.article_id, sort=True)
    rare = frame.groupby("article_id").query_id.transform("nunique").le(2).to_numpy()
    nodes = len(queries) + len(documents) + len(articles)
    rows = np.concatenate([q, q[rare]])
    cols = np.concatenate([d + len(queries), a[rare] + len(queries) + len(documents)])
    graph = coo_matrix((np.ones(len(rows), dtype=np.int8), (rows, cols)), shape=(nodes, nodes))
    return connected_components(graph, directed=False)[1][q]


def lexical_oof(train, features, folds=5):
    """Честные (out-of-fold) лексические скоры для всего train."""
    oof = np.zeros(len(train))
    for fit, val in GroupKFold(folds).split(train, groups=topic_groups(train)):
        oof[val] = train_lexical_model(train.iloc[fit], None, features.iloc[fit], features.iloc[val])
        print(f"лексика OOF: {len(val):,} строк", flush=True)
    return pd.Series(oof, index=train.index)


def stack_rows(train, count=STACK_QUERIES):
    ids = pd.Series(sorted(train.query_id.unique())).sample(min(count, train.query_id.nunique()), random_state=2026)
    return train[train.query_id.isin(set(ids))]


def stack_chunks(train, total=STACK_QUERIES, base=1200, chunk=400):
    """Первые `base` запросов совпадают с прежней стек-выборкой (их нейро-скоры уже в кеше),
    остальные считаются порциями по `chunk` запросов: каждая порция кешируется и дочитывается после обрыва."""
    total = total or train.query_id.nunique()
    base = min(base, total)
    first = stack_rows(train, base)
    rest = pd.Series(sorted(set(train.query_id) - set(first.query_id))).sample(frac=1, random_state=2026).iloc[:max(0, total - base)]
    return [first] + [train[train.query_id.isin(set(rest.iloc[i:i + chunk]))] for i in range(0, len(rest), chunk)]


def listwise(frame, scores, name):
    """Контекст кандидата внутри запроса и статьи: ранг, отрыв от лидера, уверенность, соседи по статье."""
    s = pd.Series(np.asarray(scores, dtype=np.float64), index=frame.index)
    q, qa = frame.query_id, [frame.query_id, frame.article_id]
    z = pd.Series(within_query(frame, s), index=frame.index)
    group = s.groupby(q)
    prob = np.exp(s - group.transform("max"))
    out = {
        f"{name}_raw": s, f"{name}_z": z, f"{name}_rank": group.rank(pct=True),
        f"{name}_gap": group.transform("max") - s, f"{name}_qmax": group.transform("max"),
        f"{name}_qstd": group.transform("std").fillna(0), f"{name}_prob": prob / prob.groupby(q).transform("sum"),
        f"{name}_hi": (z > 1).groupby(q).transform("sum"),
        f"{name}_a_max": z.groupby(qa).transform("max"), f"{name}_a_mean": z.groupby(qa).transform("mean"),
        f"{name}_a_hi": (z > 0.5).groupby(qa).transform("sum"),
    }
    out[f"{name}_a_gap"] = out[f"{name}_a_max"] - z
    out[f"{name}_a_rank"] = s.groupby(qa).rank(pct=True)
    out[f"{name}_a_cnt"] = s.groupby(qa).transform("size")
    return pd.DataFrame(out, index=frame.index)


def prf_similarity(frame, lexical, z):
    """Похожесть фрагмента на фрагменты запроса, которые нейросеть считает релевантными (символьный TF-IDF)."""
    x = lexical[1].transform(frame["text"].map(clean))
    z = np.asarray(z, dtype=np.float64)
    out = np.zeros((len(frame), 3), dtype=np.float32)
    for rows in frame.groupby("query_id", sort=False).indices.values():
        if len(rows) < 2:
            continue
        sim = (x[rows] @ x[rows].T).toarray()
        np.fill_diagonal(sim, 0)
        weight = np.tile(np.exp(1.5 * (z[rows] - z[rows].max())), (len(rows), 1))
        np.fill_diagonal(weight, 0)
        out[rows, 0] = (sim * weight / weight.sum(1, keepdims=True)).sum(1)
        top = np.argsort(-z[rows])[:3]
        count = np.full(len(rows), len(top)); count[top] -= 1
        out[rows, 1] = sim[:, top].sum(1) / np.maximum(count, 1)
        out[rows, 2] = sim.max(1)
    return pd.DataFrame(out, columns=["prf_weighted", "prf_top3", "prf_max"], index=frame.index)


def blend_z(frame, scores):
    """Сводный z-скор энкодеров (jina с двойным весом) — для PRF."""
    weights = {name: (2. if name == "jina" else 1.) for name in scores}
    return sum(weights[n] * within_query(frame, scores[n]) for n in scores) / sum(weights.values())


def stack_features(frame, x, scores, lexical_score, lexical):
    parts = [listwise(frame, scores[name], name) for name in scores]
    lex_z = within_query(frame, lexical_score)
    blend = blend_z(frame, scores)
    extra = pd.DataFrame({"lex_raw": np.asarray(lexical_score), "lex_z": lex_z, "blend_z": blend}, index=frame.index)
    prf = prf_similarity(frame, lexical, blend)
    prf_jina = prf_similarity(frame, lexical, within_query(frame, scores["jina"])).add_suffix("_jina")
    return pd.concat(parts + [extra, prf, prf_jina, x.loc[frame.index, STACK_FEATURES]], axis=1)


def stacker_predict(fit_frame, fit_x, target_frame, target_x, seed):
    """CatBoostRanker (YetiRank, early stopping по NDCG@10 на отложенных темах) и CatBoostClassifier.
    Возвращает z-скоры внутри запроса: (ранкер, классификатор)."""
    groups_all = topic_groups(fit_frame)
    train_ix, stop_ix = next(GroupShuffleSplit(1, test_size=.15, random_state=seed).split(fit_frame, groups=groups_all))
    def pool(ix):
        part = fit_frame.iloc[ix]; order = np.argsort(part.query_id.to_numpy(), kind="stable")
        return Pool(fit_x.iloc[ix].iloc[order], part.relevance.to_numpy()[order], group_id=part.query_id.to_numpy()[order])
    common = dict(depth=5, learning_rate=.05, l2_leaf_reg=6, thread_count=3, random_seed=seed, allow_writing_files=False, verbose=False)
    ranker = CatBoostRanker(loss_function="YetiRank", eval_metric="NDCG:top=10", iterations=1500,
                            early_stopping_rounds=120, use_best_model=True, **common)
    ranker.fit(pool(train_ix), eval_set=pool(stop_ix))
    classifier = CatBoostClassifier(iterations=500, **common).fit(Pool(fit_x, fit_frame.relevance.to_numpy()))
    return (within_query(target_frame, ranker.predict(target_x)),
            within_query(target_frame, classifier.predict(target_x, prediction_type="RawFormulaVal")))


def validate_stack(sub, feats, x_sub, scores, lex, folds=5):
    """Кросс-валидация по темам: оценка стекера и подбор весов смеси (классификатор, исходный ансамбль)."""
    rank, clf, base = (np.zeros(len(sub)) for _ in range(3))
    for number, (fit, val) in enumerate(GroupKFold(folds).split(sub, groups=topic_groups(sub))):
        fit_frame, val_frame = sub.iloc[fit], sub.iloc[val]
        rank[val], clf[val] = stacker_predict(fit_frame, feats.iloc[fit], val_frame, feats.iloc[val], STACK_SEEDS[0])
        correction = fit_jina_calibration(fit_frame, x_sub.iloc[fit], scores["jina"][fit])
        base[val] = within_query(val_frame, calibrated_ensemble(val_frame, x_sub.iloc[val], scores["jina"][val], scores["mini"][val], lex[val], correction))
        print(f"фолд {number}: ранкер {mean_ndcg(val_frame, rank[val]):.4f} | классификатор {mean_ndcg(val_frame, clf[val]):.4f} | исходный ансамбль {mean_ndcg(val_frame, base[val]):.4f}", flush=True)
    single = {f"энкодер {n}": mean_ndcg(sub, within_query(sub, v)) for n, v in scores.items()}
    single.update({"исходный ансамбль": mean_ndcg(sub, base), "CatBoostRanker": mean_ndcg(sub, rank), "CatBoostClassifier": mean_ndcg(sub, clf)})
    print("OOF NDCG@10:", {k: round(v, 4) for k, v in single.items()})
    grid = {(wc, w): mean_ndcg(sub, w * ((1 - wc) * rank + wc * clf) + (1 - w) * base) for wc in (0, .5, .75, 1) for w in (0, .5, .75, 1)}
    print("OOF смеси (вес классификатора внутри стекера, вес стекера):", {k: round(v, 4) for k, v in grid.items()})
    return max(grid, key=grid.get), grid


In [ ]:

train, test, sample = load_data()
print(platform.platform(), platform.python_version())
print(f"Train: {len(train):,} пар, {train.query_id.nunique():,} запросов")
print(f"Test: {len(test):,} пар, {test.query_id.nunique():,} запросов")
lexical = fit_lexical(train)
x_train = rank_features(train, lexical)
x_test = rank_features(test, lexical)
lexical_score = train_lexical_model(train, test, x_train, x_test)

batch_sub = {"jina": 12}
scores_test = {"jina": encoder_scores("jina", test), "mini": encoder_scores("mini", test)}
for name in [n for n in os.environ.get("AMML_EXTRA_ENCODERS", "dity,modern").split(",") if n.strip()]:
    try:
        scores_test[name] = encoder_scores(name, test)
    except Exception as error:
        print(f"энкодер {name} пропущен: {error}")
print("Энкодеры:", list(scores_test))

chunks = stack_chunks(train)
sub = pd.concat(chunks)
t0 = time.time()
scores_sub = {name: np.concatenate([encoder_scores(name, part, batch_size=batch_sub.get(name)) for part in chunks]) for name in scores_test}
print(f"нейро-скоры стек-выборки: {len(sub):,} пар, {time.time()-t0:.0f} с")
lex_oof = lexical_oof(train, x_train)
lex_sub = lex_oof.loc[sub.index].to_numpy()
x_sub = x_train.loc[sub.index]

feats_sub = stack_features(sub, x_sub, scores_sub, lex_sub, lexical)
feats_test = stack_features(test, x_test, scores_test, lexical_score, lexical)
(clf_weight, weight), grid = validate_stack(sub, feats_sub, x_sub, scores_sub, lex_sub)
print("Вес классификатора внутри стекера:", clf_weight, "| вес стекера в итоговой смеси:", weight)


parts = [stacker_predict(sub, feats_sub, test, feats_test, seed) for seed in STACK_SEEDS]
stack_test = np.mean([(1 - clf_weight) * r + clf_weight * c for r, c in parts], axis=0)
correction = fit_jina_calibration(sub, x_sub, scores_sub["jina"])
base_test = within_query(test, calibrated_ensemble(test, x_test, scores_test["jina"], scores_test["mini"], lexical_score, correction))
prediction = weight * within_query(test, stack_test) + (1 - weight) * base_test
submission = write_submission(test, sample, prediction)
submission.head()


Windows-11-10.0.22631-SP0 3.13.5
Train: 31,530 пар, 3,042 запросов
Test: 13,507 пар, 1,305 запросов
jina: скоры из amml_pre\scores_jina.npz
mini: скоры из amml_pre\scores_mini.npz
dity: готового ONNX нет, считаю через PyTorch


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

c:\Users\keles\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\keles\.cache\huggingface\hub\models--DiTy--cross-encoder-russian-msmarco. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\keles\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hu

model.safetensors: reconstructing file:   0%|          |  0.00B /  711MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

... (8 outputs left out)

dity (torch): 16/4,150
dity (torch): 1,040/4,150
dity (torch): 2,064/4,150
dity (torch): 3,088/4,150
dity (torch): 4,112/4,150
dity (torch): 4,150/4,150
dity: готового ONNX нет, считаю через PyTorch


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

dity (torch): 16/4,136
dity (torch): 1,040/4,136
dity (torch): 2,064/4,136
dity (torch): 3,088/4,136
dity (torch): 4,112/4,136
dity (torch): 4,136/4,136
нейро-скоры стек-выборки: 24,866 пар, 1543 с
лексика OOF: 8,842 строк
лексика OOF: 5,672 строк
лексика OOF: 5,672 строк
лексика OOF: 5,673 строк
лексика OOF: 5,671 строк
фолд 0: ранкер 0.9595 | классификатор 0.9656 | исходный ансамбль 0.9612
фолд 1: ранкер 0.9529 | классификатор 0.9542 | исходный ансамбль 0.9562
фолд 2: ранкер 0.9431 | классификатор 0.9498 | исходный ансамбль 0.9463
фолд 3: ранкер 0.9597 | классификатор 0.9606 | исходный ансамбль 0.9615
фолд 4: ранкер 0.9558 | классификатор 0.9590 | исходный ансамбль 0.9524
OOF NDCG@10: {'энкодер jina': np.float64(0.953), 'энкодер mini': np.float64(0.8477), 'энкодер dity': np.float64(0.8031), 'исходный ансамбль': np.float64(0.9555), 'CatBoostRanker': np.float64(0.9542), 'CatBoostClassifier': np.float64(0.9578)}
OOF смеси (вес классификатора внутри стекера, вес стекера): {(0, 0): np.flo

,id,prediction
0,r_00100bd94cf8513c6500,0.569584
1,r_0011d1a81cda1b1c720f,-0.685564
2,r_00135550b108ebd88432,-0.149391
3,r_001cac3cb24c86e57ef0,-1.332686
4,r_001e8de6d4886f85b86e,-0.007693
